In [4]:
import awkward as ak
import uproot
import numpy as np
import pandas as pd
import os
import json

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

import sys
sys.path.append("..")
from src.processors.Processor import Processor

# Dataset

In [5]:
base = '/data/pubfs/fudawei/mc'
basedir = {d: os.path.join(base, d) for d in os.listdir(base) if d=='GJets' or d=='ZpToHGamma'}
basedir

{'GJets': '/data/pubfs/fudawei/mc/GJets',
 'ZpToHGamma': '/data/pubfs/fudawei/mc/ZpToHGamma'}

In [6]:
samples = {s: [] for s in basedir}
for s in basedir:
    for (current_path, dirs, files) in os.walk(basedir[s]):
        for f in files:
            if f.endswith('.root'):
                samples[s].append(os.path.join(current_path, f))
samples

{'GJets': ['/data/pubfs/fudawei/mc/GJets/HT-40To100/3B93385F-77FD-6B47-BF83-8476E32C7515.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/B7ED4838-B32C-E347-BD38-0FE7F990A127.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/42A269A9-5AB5-1045-B806-EE5D4588043B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/5C460723-1125-3540-8FDE-472EF4ED064C.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/A7BEFB69-C329-C445-AA66-6758B5AD5CF4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/59666184-B2AA-464C-8946-7384CB7F3626.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/7B14B631-E17C-FF41-B1B1-8AF5F9EE69D0.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/E4DB23B8-2ECA-2147-A8F1-BB5C1FCBCCC4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/ED0DC548-72E5-D146-BD08-CC494A7BE30B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/1D64B0C2-65B4-BE4D-9478-4ED80332AB41.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/070AD8C7-183E-EE41-BB0F-04B720E8FBAB.root',
  '/data/pubfs/fudawei/mc/GJets/HT

# Batch test

In [4]:
_events = NanoEventsFactory.from_root(file=samples['ZpToHGamma'][0], treepath='Events', schemaclass=NanoAODSchema,).events()
p=Processor(outdir='../output/test/')
p.process(_events)

KeyboardInterrupt: 

# Parallel computing

In [7]:
cutflow = {}

In [9]:
cutflow['ZpToHGamma'] = processor.run_uproot_job(
    fileset={'ZpToHGamma': samples['ZpToHGamma']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('../output/ZpToHGamma')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 12}, # running on $workers cpu cores
)

Output()

Output()

In [7]:
cutflow

{'ZpToHGamma': {'raw': 736000,
  'electron': 317369,
  'muon': 339720,
  'photon': 278180,
  'AK8jet': 184982,
  'trigger': 661355,
  'b-veto': 358691}}

In [10]:
cutflow['GJets'] = processor.run_uproot_job(
    fileset={'GJets': samples['GJets']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('..', 'output', 'GJets')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 36}, # running on $workers cpu cores
)

Preprocessing:   0%|          | 0/137 [00:00<?, ?file/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:255: UserWarning: Cancelling 73 running jobs (likely due to an exception)
  warnings.warn(


OSError: expected Chunk of length 392,
received 0 bytes from MemmapSource
for file path /data/pubfs/fudawei/samples/mc/2018/GJets/nanov9/GJets_HT-600ToInf_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/F7D6B6E3-47C2-BD4D-A467-434C36B28B4F.root

In [7]:
cutflow['QCD'] = processor.run_uproot_job(
    fileset={'QCD': samples['QCD']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('..', 'output', 'GJets')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 36}, # running on $workers cpu cores
)

Preprocessing:   0%|          | 0/704 [00:00<?, ?file/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:255: UserWarning: Cancelling 73 running jobs (likely due to an exception)
  warnings.warn(


OSError: expected Chunk of length 392,
received 0 bytes from MemmapSource
for file path /data/pubfs/fudawei/samples/mc/2018/QCD/nanov9/QCD_HT500to700_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/CCCEE5F3-3F4F-2145-B66F-F7F07879678C.root